# Pulling prices & earnings dates

### Aim

The aim of this notebook is to succesfully pull ohlcv data and earnings dates for our tickers so that it can be used in the future for creating labels and features.

**Output:** `raw_ohlcv.parquet` & `raw_earnings.parquet`

### Setup

To being with, let's import the necessary libraries needed for this notebook.

In [69]:
import yfinance as yf
import pandas as pd
import datetime as dt
from stock_predictor.config import PRIMARY_TICKER, NEWS_START_DATE, NEWS_END_DATE, RAW_DATA_DIR

print(f"Primary ticker: {PRIMARY_TICKER}")
print(F"Article Publication Timeframe: {NEWS_START_DATE} to {NEWS_END_DATE}")

Primary ticker: TSLA
Article Publication Timeframe: 2025-08-01 to 2026-08-01


### Pulling prices & earnings dates

We want to start by collecting market data. Since we will need some data before and after the publication dates for our features and labels (e.g. abnormal_return_1d, volatility_20d), we should add 40 days either side of the range so that we will comfortably have enough data. yFinance allows us to pass multiple tickers at once and pull data for them all into one dataframe, which we will include here.

In [70]:
def pull_ohlcv(start_date, end_date, tickers):
    """
    Return company daily ohlcv within the given range as a dataframe.

    tickers: symbols for which we pull the data
    start_date, end_date: bounds for the publication dates of the articles
    """
    #Add days to extend our range for extra data
    price_start = pd.Timestamp(start_date) - dt.timedelta(days=40)
    price_end = pd.Timestamp(end_date) + dt.timedelta(days=40)

    ohlcv_df = yf.download(tickers, start=price_start, end=price_end)
    return ohlcv_df

We also need to collect the earnings dates. Since yFinance doesnt allow to filter by date, we need to do this ourselves after calling the function, which returns 12 by default, including future ones as well as the past ones. We only restrict the start date, since future earnings dates are necessary to compute the days to the next earnings call, which is one of our features.

In [71]:
def pull_earnings_dates(start_date, ticker):
    """
    Return company eranings dates within the given range as a dataframe.

    ticker: symbol for which we pull the data
    start_date: start bound for the publication dates of the articles
    """
    earnings = yf.Ticker(ticker).get_earnings_dates()
    #Filter earnings dates
    earnings = earnings[(earnings.index >= start_date)]
    return earnings

Let's now pull this data for **Tesla** (symbol "TSLA"), our primary ticker, as well as ohlcv for symbol SPY, which will be needed for calcualating abnormal returns later on. Since we are pulling data for around a year and 2 months, we are expecting data for around 60 weeks, not including weekends and other holidays, which should be around 280 days.

In [72]:
tsla_spy_ohlcv = pull_ohlcv(NEWS_START_DATE, NEWS_END_DATE, ["TSLA", "SPY"])
print(tsla_spy_ohlcv.count())
tsla_spy_ohlcv.head()


[*********************100%***********************]  2 of 2 completed

Price   Ticker
Close   SPY       284
        TSLA      284
High    SPY       284
        TSLA      284
Low     SPY       284
        TSLA      284
Open    SPY       284
        TSLA      284
Volume  SPY       284
        TSLA      284
dtype: int64


Price            Close                    High                     Low  \
Ticker             SPY        TSLA         SPY        TSLA         SPY   
Date                                                                     
2025-06-23  593.573059  348.679993  593.958739  357.540009  585.403570   
2025-06-24  600.130371  340.470001  601.188592  356.260010  596.797247   
2025-06-25  600.466614  327.549988  601.940275  343.000000  598.903912   
2025-06-26  605.164612  325.779999  605.599792  331.049988  601.702968   
2025-06-27  608.171204  323.630005  609.635025  329.339996  604.135960   

Price                         Open                Volume             
Ticker            TSLA         SPY        TSLA       SPY       TSLA  
Date                                                                 
2025-06-23  327.480011  588.519013  327.540009  87426000  190716800  
2025-06-24  340.440002  597.707208  356.170013  67735300  114736200  
2025-06-25  320.399994  601.247934  342.700012  62114800  119845100  
2025-06-26  323.609985  602.316168  324.609985  78548400   80440900  
2025-06-27  317.500000  606.163482  324.510010  86258400   89067000

Let's make sure our data is complete.

In [73]:
tsla_spy_ohlcv.isna().sum()

Price   Ticker
Close   SPY       0
        TSLA      0
High    SPY       0
        TSLA      0
Low     SPY       0
        TSLA      0
Open    SPY       0
        TSLA      0
Volume  SPY       0
        TSLA      0
dtype: int64

Now let's pull the earnings calendar for Tesla.

In [74]:
tsla_earnings = pull_earnings_dates(NEWS_START_DATE, "TSLA")
print(tsla_earnings)

                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-10-21 16:00:00-04:00          0.46           NaN          NaN
2026-07-22 16:00:00-04:00          0.54          0.33       -38.35
2026-04-22 16:00:00-04:00          0.35          0.41        17.15
2026-01-28 16:00:00-05:00          0.45          0.50        10.96
2025-10-22 16:00:00-04:00          0.56          0.50       -10.53


### Saving our data

Since all of our data is valid, we can now save this for future use.

In [75]:
tsla_spy_ohlcv.to_parquet(RAW_DATA_DIR / "raw_ohlcv.parquet")
tsla_earnings.to_parquet(RAW_DATA_DIR / "raw_earnings.parquet")